### Prepare the PCA dataset

Prepare the flow measurements for PCA and keep their labels and timestamps for Section 2 of the assignment.


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.preprocessing import DropAllZeroColumns, SpearmanFilter

In [ ]:
df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip()
flows = df.set_index("Unnamed: 0").drop(
    columns=["Flow ID", "Source Port", "Destination Port"]
)

### Choose the PCA inputs

The original row ID keeps each flow aligned with its label and timestamp. `Flow ID` duplicates information in other columns. The prepared dataset also omits the source and destination ports.

PCA uses measurements and counts. `Inbound` and protocol indicators such as `Protocol_UDP` represent categories and stay with the metadata for inspection.


In [ ]:
# Protocol numbers from the IANA registry.
# https://www.iana.org/assignments/protocol-numbers/protocol-numbers.xhtml
protocol_names = {0: "HOPOPT", 6: "TCP", 17: "UDP"}
flows["Protocol"] = (
    flows["Protocol"]
    .map(protocol_names)
    .fillna("Unknown_" + flows["Protocol"].astype(str))
)
flows = pd.get_dummies(flows, columns=["Protocol"], dtype=int)

categorical_columns = {"label", "Inbound"}
measurement_columns = [
    column
    for column in flows.select_dtypes(include="number").columns
    if column not in categorical_columns and not column.startswith("Protocol_")
]
X = flows[measurement_columns]
metadata = flows.drop(columns=measurement_columns)
y = metadata["label"].copy()

### Define the preprocessing pipeline

1. Drop measurements that are zero in every row used to fit the pipeline.
2. Scan the remaining measurements in column order. Drop a measurement if its absolute Spearman correlation with any retained measurement exceeds 0.9. For example, if two columns have a correlation of -0.95, keep the first.
3. Standardize the retained measurements so their units do not dominate PCA.
4. Keep enough principal components to explain at least 99% of the variance in the standardized measurements.
5. Name the component columns `PC1`, `PC2`, and so on. Pandas output carries the original row IDs through scaling and PCA.


In [ ]:
def name_components(scores):
    names = [f"PC{i}" for i in range(1, scores.shape[1] + 1)]
    return scores.set_axis(names, axis="columns")


preprocess = Pipeline(
    [
        ("drop_all_zero", DropAllZeroColumns()),
        ("spearman", SpearmanFilter(threshold=0.9)),
        ("scale", StandardScaler().set_output(transform="pandas")),
        (
            "pca",
            PCA(n_components=0.99, svd_solver="full").set_output(transform="pandas"),
        ),
        ("name_components", FunctionTransformer(name_components)),
    ]
)

### Inspect the exploratory fit

Fit on all flows to inspect the feature removals and retained variance. `X_reduced` contains the components. `df_pca` joins them to the label and other metadata by row ID for analysis.


In [ ]:
X_reduced = preprocess.fit_transform(X)
df_pca = pd.concat([X_reduced, metadata], axis=1)

In [ ]:
columns_to_drop = preprocess.named_steps["drop_all_zero"].dropped_columns_
correlation_removals = preprocess.named_steps["spearman"].removals_
pca = preprocess.named_steps["pca"]
variance_retained = pca.explained_variance_ratio_.sum()

print(f"All-zero columns removed: {', '.join(columns_to_drop)}")
print(f"{pca.n_components_} components retain {variance_retained:.2%} of variance.")
correlation_removals

### Starting Section 2

Split the original measurements in `X` with the aligned labels in `y`. Use the timestamps in `metadata` to order the flows within each label for the temporal split.

For each split, fit a fresh copy of the pipeline on the training measurements and transform the test measurements with that fitted copy. This keeps test flows from influencing which columns we retain, how we scale them, or how PCA combines them.
